# 0.2 · NumPy Deep Dive 全面解析

> **课程定位 / Where this fits**
> 本 notebook 是 `DataScience-from-scratch` 的第 2 课，**Part 0 · 基础准备**。
> 上一课讲的是 Python **语言本身**；这一课讲数据科学栈真正的"地基"——`numpy.ndarray`。
> Lesson 2 of `DataScience-from-scratch`, **Part 0 · Foundations**.
> Previous lesson covered Python the language; this one covers the actual foundation of the DS stack — `numpy.ndarray`.

> 📐 **符号约定 / Notation** （详见仓库根目录 [`NOTATION.md`](../NOTATION.md)）
> - $n$：样本数 / number of samples
> - $d$：特征数 / number of features
> - $\mathbf{x} \in \mathbb{R}^d$：向量（粗体小写）/ vector (bold lowercase)
> - $\mathbf{X} \in \mathbb{R}^{n\times d}$：矩阵（粗体大写）/ matrix (bold uppercase)
> - $\mathbf{X}^\top$：转置 / transpose
> - $\mathbf{X}^{-1}$：逆 / inverse
> - $\|\mathbf{x}\| \equiv \|\mathbf{x}\|_2$：默认 L2 范数 / default L2 norm

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook, you'll be able to:

1. 解释 `ndarray` 的内存布局：`shape`、`dtype`、`strides`、`itemsize`、C-order vs F-order。
   Explain `ndarray` memory layout: shape, dtype, strides, itemsize, C-order vs F-order.
2. 熟练用**向量化**写出比纯 Python 快 **100–1000 倍**的代码。
   Vectorize code to be **100–1000×** faster than pure Python.
3. 准确预测**广播**（broadcasting）的输出 shape。
   Predict the output shape of any **broadcasting** operation.
4. 用 `numpy.linalg` 求解线性方程组、特征分解、SVD。
   Use `numpy.linalg` for linear systems, eigen-decomposition, SVD.
5. 用 `einsum` 简洁表达任意张量运算。
   Express arbitrary tensor operations with `einsum`.
6. 在 **California Housing** 数据集上**只用 NumPy** 解析地求出线性回归的权重（正规方程）。
   Solve linear regression on **California Housing** **using only NumPy** (normal equation).

---

## 目录 / Table of Contents

1. [为什么需要 NumPy / Why NumPy](#1)
2. [`ndarray` 解剖：shape / dtype / strides](#2)
3. [创建数组 / Creating Arrays](#3)
4. [索引与切片 / Indexing & Slicing](#4)
5. [高级索引：fancy + boolean / Advanced Indexing](#5)
6. [视图 vs 拷贝（必须懂的坑）/ Views vs Copies](#6)
7. [广播 / Broadcasting](#7)
8. [通用函数 / Universal Functions (ufuncs)](#8)
9. [聚合与轴 / Aggregations & axis](#9)
10. [形状操作 / Reshape, Transpose, Stack, Concatenate](#10)
11. [线性代数 / Linear Algebra](#11)
12. [随机数生成 / Random Number Generation](#12)
13. [`einsum` —— 张量运算瑞士军刀](#13)
14. [性能小贴士 / Performance Tips](#14)
15. [实战：用 NumPy 在 California Housing 上做线性回归 / Hands-on](#15)
16. [小结 / Summary](#16)


<a id="1"></a>
## 1. 为什么需要 NumPy / Why NumPy

Python 的 `list` 灵活但**慢得离谱**：每个元素都是一个独立的 Python 对象，遍历要不断 unbox/box。
Python `list`s are flexible but **painfully slow**: every element is a separate Python object with unbox/box overhead.

`numpy.ndarray` 不一样：
- **同构** / **homogeneous**：所有元素 dtype 一致
- **连续内存** / **contiguous memory**：一片紧凑的 C 数组
- **底层 BLAS / SIMD** / **native BLAS / SIMD**：调用 C / Fortran 库做向量化

数学上，NumPy 就是把 Python 变成了一门**面向数组的语言**——一行代码做一整个向量/矩阵的运算：

$$
\mathbf{z} = \mathbf{x} + \mathbf{y}, \quad \mathbf{x}, \mathbf{y} \in \mathbb{R}^d
\quad\Longrightarrow\quad \texttt{z = x + y}
$$


In [ ]:
# 性能对比 / Benchmark
import time
import numpy as np

n = 10_000_000

# 1) 纯 Python 列表 / pure Python list
xs = list(range(n))
t0 = time.perf_counter()
ys = [x * 2 + 1 for x in xs]
t_py = time.perf_counter() - t0

# 2) NumPy 向量化 / vectorized
arr = np.arange(n)
t0 = time.perf_counter()
arr_y = arr * 2 + 1
t_np = time.perf_counter() - t0

print(f"pure python : {t_py*1000:7.1f} ms")
print(f"numpy        : {t_np*1000:7.1f} ms")
print(f"speedup      : ×{t_py / t_np:.0f}")


典型加速 **30–200 倍**（具体取决于 CPU 和数据量）。等我们进入到 ML 算法，**没有 NumPy 等于没有 ML**。
Typical speedup is **30–200×**, depending on hardware. **No NumPy ⇒ no practical ML.**


<a id="2"></a>
## 2. `ndarray` 解剖 / Anatomy of an `ndarray`

每个数组都有 4 个核心属性：
Every array has four core attributes:

| 属性 / Attribute | 含义 / Meaning |
|---|---|
| `shape` | 每个维度的大小 / size of each dim, e.g. `(3, 4)` |
| `dtype` | 元素类型 / element type, e.g. `float64` |
| `ndim` | 维度数 / number of dims |
| `strides` | 沿每个维度走一步占多少字节 / bytes to step per dim |

这四个东西决定了**怎么从一块连续内存里读出多维数组**。
These four things determine **how a multi-dim array is read from a flat memory block**.


In [ ]:
import numpy as np

A = np.arange(12).reshape(3, 4)        # 3×4 矩阵 / 3-by-4 matrix
print(A)
print(f"shape   : {A.shape}")          # (3, 4)
print(f"dtype   : {A.dtype}")          # int64
print(f"ndim    : {A.ndim}")           # 2
print(f"itemsize: {A.itemsize} bytes") # 8 (int64 = 8 bytes)
print(f"nbytes  : {A.nbytes} bytes")   # 12 * 8 = 96
print(f"strides : {A.strides}")        # (32, 8) — 行步长 32B, 列步长 8B


**`strides` 是怎么回事？/ What are strides?**

`(32, 8)` 的意思：
- 行方向走一步（i → i+1）跳 32 字节（= 4 × 8，因为一行有 4 个 int64）
- 列方向走一步（j → j+1）跳 8 字节

这就是**为什么 transpose 不复制数据**——它只是把 `strides` 调换一下：
This is **why transpose doesn't copy data** — it just swaps the strides:


In [ ]:
print("A.T shape  :", A.T.shape)
print("A.T strides:", A.T.strides)     # 反过来 / reversed
# A 和 A.T 共享同一块内存 / A and A.T share the same buffer
print("same data? :", A.T.base is A)   # True


**C-order vs F-order**

- **C-order**（NumPy 默认）：最右侧维度变化最快 → 行优先
- **F-order**（Fortran / MATLAB 默认）：最左侧维度变化最快 → 列优先

实际工作里**几乎一直用 C-order**；但和 R / MATLAB / 某些 GPU 库交互时要小心。
In practice **almost always C-order**; only watch out when interfacing with R / MATLAB / some GPU libs.


In [ ]:
C_arr = np.arange(6).reshape(2, 3)           # 默认 C-order
F_arr = np.asfortranarray(C_arr)             # 转 F-order
print("C strides:", C_arr.strides)           # (24, 8)
print("F strides:", F_arr.strides)           # (8, 16)


**dtype 速查 / dtype Cheat Sheet**

| dtype | 字节 / Bytes | 备注 / Notes |
|---|---|---|
| `int8` / `int16` / `int32` / `int64` | 1/2/4/8 | 有符号整数 / signed |
| `uint8` / ... / `uint64` | 1–8 | 无符号 / unsigned；图像常用 `uint8` |
| `float16` / `float32` / `float64` | 2/4/8 | DL 训练常用 `float32`；推理常用 `float16` |
| `bool` | 1 | 布尔 |
| `complex64` / `complex128` | 8/16 | 复数 |

**省内存的常见操作 / Common memory savings**:
- 大数据从 `float64` 降到 `float32` → 内存减半
- 类别 ID 从 `int64` 降到 `int32` 甚至 `int16`


In [ ]:
x64 = np.arange(1_000_000, dtype=np.float64)
x32 = x64.astype(np.float32)
print(f"float64: {x64.nbytes/1024/1024:.2f} MB")
print(f"float32: {x32.nbytes/1024/1024:.2f} MB")


<a id="3"></a>
## 3. 创建数组 / Creating Arrays

几乎所有 NumPy 代码都从"先造一个数组"开始。下面是工业级最常用的几种方式。
Most NumPy code starts by creating an array. Here are the industrial-strength options.


In [ ]:
# 从 Python 列表 / from Python list
a = np.array([[1, 2, 3], [4, 5, 6]])
print("from list:\n", a)

# 全 0 / 全 1 / 单位阵 / 全某值
print("zeros:\n", np.zeros((2, 3)))
print("ones :\n", np.ones((2, 3), dtype=np.int32))
print("eye  :\n", np.eye(3))                  # 单位阵 / identity matrix
print("full :\n", np.full((2, 3), 7.5))       # 全 7.5


In [ ]:
# 等差 / 等分点
print("arange    :", np.arange(0, 10, 2))                # [0, 2, 4, 6, 8]
print("linspace  :", np.linspace(0, 1, 5))               # 5 个等分点 [0, .25, .5, .75, 1]
print("logspace  :", np.logspace(0, 3, 4))               # [1, 10, 100, 1000]


In [ ]:
# 随机数 —— 推荐用"新 API" Generator
# Prefer the new-API Generator (np.random.default_rng) over the legacy np.random.*
rng = np.random.default_rng(seed=42)

print("uniform [0,1):\n", rng.random((2, 3)))
print("normal N(0,1):\n", rng.standard_normal((2, 3)))
print("int [0, 10) :\n", rng.integers(0, 10, size=(2, 3)))


> **`np.random.rand` vs `default_rng` / Legacy vs new API**
> 老 API（`np.random.seed`, `np.random.rand`...）是全局状态，**不可重入**。新 API `default_rng()` 返回独立的 `Generator` 对象，多线程/并行安全。**所有新代码请用 `default_rng()`**。
> The legacy global-state API is not re-entrant. The new `Generator` API is thread/parallel-safe. **Use `default_rng()` for new code.**


<a id="4"></a>
## 4. 索引与切片 / Indexing & Slicing

NumPy 的索引语法**比 Python 强大得多**——你可以一行选一整行、一整列、一个子矩阵。
NumPy indexing is **much more powerful than Python** — pick rows, columns, sub-matrices in one line.


In [ ]:
A = np.arange(20).reshape(4, 5)
print("A =\n", A)

print("\nA[1, 2]   =", A[1, 2])     # 单元素 / scalar
print("A[1]      =", A[1])          # 第 1 行 / row 1
print("A[:, 2]   =", A[:, 2])       # 第 2 列 / col 2
print("A[1:3, 2:5]:\n", A[1:3, 2:5])  # 子矩阵 / sub-matrix


**和 Python list 的区别 / Difference from Python list**

> `A[1, 2]` 是 NumPy 专属！Python `list` 只能写 `lst[1][2]`。
> `A[1, 2]` is NumPy-only. Python lists need `lst[1][2]`.
>
> 用 `A[1, 2]` 比 `A[1][2]` **更快**（少一次中间数组创建）。
> `A[1, 2]` is **faster** than `A[1][2]` (avoids creating an intermediate array).


In [ ]:
# 三个点 / 三个冒号特殊用法 / Special slicing
B = np.arange(24).reshape(2, 3, 4)
print("B shape:", B.shape)

# 反转某一维 / reverse along a dim
print("\nreverse rows of slice 0:\n", B[0, ::-1])

# 省略号 / Ellipsis: "其余所有维度"
print("\nB[..., 0] (last dim first elem):\n", B[..., 0])

# np.newaxis (== None): 在指定位置插入新维度 —— 广播必杀技 / broadcasting trick
v = np.array([1, 2, 3])
print("\nv shape           :", v.shape)
print("v[:, np.newaxis] :", v[:, np.newaxis].shape)   # (3, 1)
print("v[np.newaxis, :] :", v[np.newaxis, :].shape)   # (1, 3)


<a id="5"></a>
## 5. 高级索引：fancy + boolean / Advanced Indexing

两个**极其常用**的高级索引方式：
Two **incredibly common** advanced-indexing modes:

- **Fancy indexing**：用整数数组当索引
- **Boolean masking**：用布尔数组当过滤器


In [ ]:
x = np.array([10, 20, 30, 40, 50])

# fancy: 用一个整数数组挑出指定位置 / pick by integer array
idx = np.array([0, 2, 4, 2])    # 可以重复 / can repeat
print("fancy   :", x[idx])      # [10 30 50 30]

# boolean: 用同形状的布尔数组当筛选 / filter by same-shape bool array
mask = x > 20
print("mask    :", mask)        # [F F T T T]
print("filtered:", x[mask])     # [30 40 50]


In [ ]:
# 二维 fancy indexing —— 注意 shape 规则
# 2-D fancy indexing — note the shape rule
A = np.arange(20).reshape(4, 5)
print("A =\n", A)

rows = np.array([0, 1, 2])
cols = np.array([1, 3, 4])
# 取出 (0,1), (1,3), (2,4) 三个元素 —— 不是 3×3 子矩阵！
# Picks three elements (0,1), (1,3), (2,4) — NOT a 3×3 sub-matrix!
print("\nA[rows, cols] =", A[rows, cols])

# 要 3×3 子矩阵的话用 np.ix_
# Use np.ix_ for the Cartesian product
print("\nA[np.ix_(rows, cols)] =\n", A[np.ix_(rows, cols)])


**Boolean masking 的实战用法 / Real-world boolean masking**

数据清洗里几乎是基本动作：把异常值/缺失值挑出来。
A basic move in data cleaning: select outliers / missing values.


In [ ]:
rng = np.random.default_rng(0)
data = rng.normal(loc=0, scale=1, size=1000)

# 找出 3-sigma 外的"异常值" / Find outliers beyond 3 sigma
mu, sigma = data.mean(), data.std()
outlier_mask = np.abs(data - mu) > 3 * sigma
n_outliers = outlier_mask.sum()             # bool 求和 = 计数 / sum of bool = count
print(f"#outliers (>3σ): {n_outliers}")
print(f"outlier values : {data[outlier_mask]}")


<a id="6"></a>
## 6. 视图 vs 拷贝 / Views vs Copies — **最常见的坑** / The #1 Gotcha

NumPy 为了快，**大多数切片操作只返回视图（view），不复制数据**。
For speed, **most slicing returns a view, not a copy**.

| 操作 / Op | 视图还是拷贝 / View or copy |
|---|---|
| 基础切片 / basic slicing `A[1:3]` | **视图** / view |
| `A.T`, `.reshape(...)` 兼容形状时 / reshape if shape-compatible | 视图 / view |
| Fancy indexing `A[[0,2]]` | **拷贝** / copy |
| Boolean mask `A[A>0]` | **拷贝** / copy |
| `A.copy()` | 拷贝 / copy |

判断方法：`arr.base is None` → 拷贝；否则是某个数组的视图。
Quick check: `arr.base is None` ⇒ copy; otherwise it's a view onto something.


In [ ]:
A = np.arange(12).reshape(3, 4)
print("A =\n", A)

# 1) 切片 → 视图：改 sub 也会改 A
sub = A[:, 1:3]
sub[:] = -1
print("\n--- 修改视图后 A 也变了 / view: A is mutated ---")
print("A =\n", A)


In [ ]:
# 2) Fancy indexing → 拷贝：改 sub 不影响 A
A = np.arange(12).reshape(3, 4)
sub = A[[0, 2], :]
sub[:] = -1
print("--- fancy indexing 是拷贝，A 不变 / fancy: A untouched ---")
print("A =\n", A)
print("\nsub =\n", sub)


**实战教训 / Real-world lesson**：在做特征工程时，如果你 `df_processed = df[df['age'] > 18]` 然后改 `df_processed`，**pandas 会大声警告**——这就是因为不知道是视图还是拷贝。NumPy 这里同理。
When doing feature engineering, `df_processed = df[df['age'] > 18]` then modifying `df_processed` triggers the famous **SettingWithCopyWarning** in pandas — same root cause as here.


<a id="7"></a>
## 7. 广播 / Broadcasting

NumPy 最有用、也最绕的特性。**它让形状不一样的数组也能做元素运算**。
NumPy's most useful (and trickiest) feature. **It lets arrays of different shapes do elementwise ops.**

### 广播规则 / The Broadcasting Rules

把两个数组的 shape **从右往左**对齐，然后逐维比较：
Align shapes from the **right**, compare each dim:

1. 如果两边相等 → OK
2. 如果其中一边是 1 → 这一维被**复制扩展**
3. 如果两边都不是 1 且不相等 → **报错**
4. 如果某边的维度数比另一边少 → 左侧补 1

例：
$$\underbrace{(\,n,\,d\,)}_{\mathbf{X}} \;\;+\;\; \underbrace{(\,d,\,)}_{\bar{\mathbf{x}}} \;\Longrightarrow\; \mathbf{X} - \bar{\mathbf{x}}$$

把均值向量从每一行减掉——**特征中心化**这种操作一行搞定。
The mean vector gets subtracted from each row — **feature centering** in one line.


In [ ]:
# 标量广播 / Scalar broadcasting
x = np.array([1, 2, 3, 4])
print("x + 10:", x + 10)   # 标量自动广播到每个元素 / scalar broadcasts to every element

# 向量 + 标量已经是广播 / vector + scalar already involves broadcasting


In [ ]:
# 经典：矩阵每一行减均值 / Classic: subtract column means from every row
rng = np.random.default_rng(0)
X = rng.normal(loc=5, scale=2, size=(5, 3))      # 5×3，5 samples × 3 features
print("X =\n", X)
print("X.shape       :", X.shape)

col_mean = X.mean(axis=0)                        # 每列均值，shape (3,)
print("\ncol_mean      :", col_mean)
print("col_mean.shape:", col_mean.shape)

# X (5,3) - col_mean (3,) → 广播后变成 (5,3) - (1,3) → (5,3)
X_centered = X - col_mean
print("\nX_centered =\n", X_centered)
print("col means now :", X_centered.mean(axis=0).round(8))   # 应该 ≈ 0


In [ ]:
# 外积 / Outer product 也是广播
a = np.array([1, 2, 3])
b = np.array([10, 20, 30, 40])
# a 变成 (3,1), b 变成 (1,4) → 输出 (3,4)
outer = a[:, None] * b[None, :]
print("outer shape:", outer.shape)
print(outer)


### 广播的"陷阱可视化" / Visual aid

```
形状 / shape   :  (5, 3)     ←  X
                +    (3,)    ←  vec
对齐(从右往左) :  (5, 3)
                  (1, 3)     ←  vec 在左侧补 1
广播后等效于   :  (5, 3)
                  (5, 3)     ←  vec 沿 axis 0 复制 5 份
```

**最常踩的坑 / Most common pitfall**：以为 `X (n, d) + y (n,)` 会沿样本维度广播。**不会！** 因为 `(n, d)` 和 `(n,)` 从右对齐变成 `(n, d)` vs `(n,)`，后者最右维 = $n \ne d$（除非碰巧相等）。
**Pitfall**: thinking `X (n, d) + y (n,)` broadcasts along samples. **It does NOT** — right-alignment gives `(n, d)` vs `(n,)`, rightmost dim mismatched.

正确做法：把 `y` 变成 `(n, 1)`：`y[:, None]`。
Fix: `y[:, None]` ⇒ shape `(n, 1)`.


In [ ]:
# 演示这个坑 / Demo the pitfall
X = np.arange(15).reshape(5, 3)    # (5, 3)
y = np.array([1, 2, 3, 4, 5])      # (5,)

try:
    X + y                          # ❌ 错的 / wrong
except ValueError as e:
    print("Error:", e)

# 正确：y 变成 (5, 1)
print("\nfixed:\n", X + y[:, None])


<a id="8"></a>
## 8. 通用函数 / Universal Functions (ufuncs)

ufunc = 逐元素作用、底层 C 实现、自动广播的函数。
ufunc = elementwise, C-implemented, broadcast-aware function.

数学上：对数组 $\mathbf{x}$，ufunc 就是对每个元素施加同一个函数 $f$，
$$ \mathbf{y}_i = f(\mathbf{x}_i) $$


In [ ]:
x = np.linspace(-2, 2, 7)
print("x         :", x)

print("\nexp(x)    :", np.exp(x))
print("log(|x|+1):", np.log(np.abs(x) + 1))
print("sqrt(x²)  :", np.sqrt(x ** 2))
print("sigmoid   :", 1 / (1 + np.exp(-x)))   # 一行写出 σ(z)=1/(1+e^{-z})

# 二元 ufunc / Binary ufuncs
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])
print("\nmaximum   :", np.maximum(a, b))
print("power     :", np.power(a, b))


<a id="9"></a>
## 9. 聚合 / Aggregations & 轴 / `axis`

**整个 DS 里 80% 的 NumPy bug 是 `axis` 搞错了。**
**80% of NumPy bugs in DS are wrong `axis` arguments.**

记忆口诀 / Mnemonic：
- `axis=0` ⇒ **沿行方向消除**，得到**列**统计量 → "down the columns"
- `axis=1` ⇒ **沿列方向消除**，得到**行**统计量 → "across the rows"

更准确：`axis=k` 表示**把第 k 个维度压扁**。
More precisely: `axis=k` means **collapse the k-th dimension**.


In [ ]:
X = np.arange(12).reshape(3, 4)
print("X =\n", X)
print("X.shape:", X.shape)

print("\nX.sum()           =", X.sum())              # 全和 / total
print("X.sum(axis=0)     =", X.sum(axis=0))         # shape (4,) - 列和 / col sums
print("X.sum(axis=1)     =", X.sum(axis=1))         # shape (3,) - 行和 / row sums
print("X.sum(axis=1, keepdims=True):\n",
      X.sum(axis=1, keepdims=True))                  # shape (3, 1) 保维度


**`keepdims=True` 的妙用 / Why `keepdims` matters**

保维度后**广播就能直接对齐**，写归一化代码不用 `[:, None]`：
With `keepdims=True`, broadcasting aligns automatically — no `[:, None]` needed for normalization:


In [ ]:
# 行归一化（每行除以行 L1 范数）/ row-normalize by L1
X = np.array([[1., 2., 3.], [4., 5., 6.]])
row_sums = X.sum(axis=1, keepdims=True)   # (2, 1) 而不是 (2,)
X_norm = X / row_sums                       # 自动广播 / auto-broadcast
print("X_norm =\n", X_norm)
print("row sums of result:", X_norm.sum(axis=1))   # 应该都是 1


In [ ]:
# 常用聚合家族 / The aggregation family
rng = np.random.default_rng(0)
A = rng.normal(size=(4, 5))
print("A =\n", A.round(2))

print("\nmean :", A.mean(axis=0).round(3))
print("std  :", A.std(axis=0).round(3))
print("min  :", A.min(axis=0).round(3))
print("max  :", A.max(axis=0).round(3))
print("argmax (per col):", A.argmax(axis=0))     # 每列最大值的行索引
print("median:", np.median(A, axis=0).round(3))
print("quantile @ 0.25:", np.quantile(A, 0.25, axis=0).round(3))


<a id="10"></a>
## 10. 形状操作 / Reshape, Transpose, Stack, Concatenate

数据科学里这些操作天天用——拼接 batch、转置矩阵、把一维变成 (n, 1)。
Used daily — concat batches, transpose matrices, turn 1-D into (n, 1).


In [ ]:
x = np.arange(12)
print("x.shape          :", x.shape)
print("reshape(3,4)     :\n", x.reshape(3, 4))
print("\nreshape(-1, 4)   :\n", x.reshape(-1, 4))    # -1 让 NumPy 自动算 / -1 = infer
print("\nx.reshape(2,2,3) shape:", x.reshape(2, 2, 3).shape)

# flatten vs ravel
A = np.arange(6).reshape(2, 3)
print("\nA.flatten() (拷贝):", A.flatten())
print("A.ravel()   (视图):", A.ravel())              # 尽量返回视图 / view if possible


In [ ]:
# 拼接 / Concatenate
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print("vstack (沿 axis 0):\n", np.vstack([a, b]))    # 垂直堆叠 / vertical
print("\nhstack (沿 axis 1):\n", np.hstack([a, b]))   # 水平堆叠 / horizontal
print("\ngeneric concat axis=0:\n", np.concatenate([a, b], axis=0))

# stack: 新增一个维度 / introduce a new dim
print("\nnp.stack shape:", np.stack([a, b]).shape)   # (2, 2, 2) — 新轴在 axis 0


In [ ]:
# 转置 / Transpose
A = np.arange(24).reshape(2, 3, 4)
print("A.shape       :", A.shape)               # (2, 3, 4)
print("A.T.shape     :", A.T.shape)             # (4, 3, 2)
print("A.transpose(1,0,2).shape:", A.transpose(1, 0, 2).shape)  # (3, 2, 4)

# swapaxes: 只交换两个轴 / swap two dims
print("A.swapaxes(0, 2).shape  :", A.swapaxes(0, 2).shape)      # (4, 3, 2)


<a id="11"></a>
## 11. 线性代数 / Linear Algebra

`numpy.linalg` 是 ML 的"瑞士军刀"。线性回归、PCA、神经网络都建立在这些操作上。
`numpy.linalg` is the Swiss army knife of ML. Linear regression, PCA, NNs all stand on these.

| 数学 / Math | NumPy | 说明 |
|---|---|---|
| $\mathbf{A}\mathbf{B}$ | `A @ B` 或 `A.dot(B)` | 矩阵乘 / matmul |
| $\mathbf{A}^\top$ | `A.T` | 转置 |
| $\mathbf{A}^{-1}$ | `np.linalg.inv(A)` | 逆（**几乎不要直接用**，见下面） |
| $\mathbf{A}\mathbf{x} = \mathbf{b}$ | `np.linalg.solve(A, b)` | 解线性方程组 ✅ 推荐 |
| 最小二乘 / least squares | `np.linalg.lstsq(A, b, rcond=None)` | 解超定方程 / overdetermined |
| 行列式 | `np.linalg.det(A)` | |
| 特征值/向量 | `np.linalg.eig(A)`, `eigh(A)` | `eigh` 用于对称矩阵 |
| SVD | `np.linalg.svd(A)` | PCA 的基石 |
| 范数 | `np.linalg.norm(x, ord=2)` | |


In [ ]:
rng = np.random.default_rng(42)
A = rng.normal(size=(3, 3))
b = rng.normal(size=3)

print("A =\n", A.round(3))
print("b =", b.round(3))

# 解 A x = b ：两种方式 / two ways
x_inv = np.linalg.inv(A) @ b           # ❌ 数值不稳定
x_solve = np.linalg.solve(A, b)         # ✅ 推荐

print("\nx via inv  :", x_inv.round(6))
print("x via solve:", x_solve.round(6))
print("close?     :", np.allclose(x_inv, x_solve))


> **为什么 `solve` 优于 `inv` / Why `solve` beats `inv`**
> 求逆是 $O(n^3)$ 且**数值不稳定**；`solve` 用 LU 分解一次性算出 $\mathbf{x}$，**速度 + 精度都更好**。
> Inversion is $O(n^3)$ and numerically shaky; `solve` does LU once, **faster and more accurate**.


In [ ]:
# 最小二乘：解 min ||A x - b||² —— 即线性回归
# Least squares: solve min ||A x - b||² — linear regression
m, d = 50, 3
A = rng.normal(size=(m, d))
true_w = np.array([1.5, -2.0, 0.5])
b = A @ true_w + 0.1 * rng.normal(size=m)    # 加点噪声 / add noise

w_hat, residuals, rank, sv = np.linalg.lstsq(A, b, rcond=None)
print("true w :", true_w)
print("w_hat  :", w_hat.round(3))


In [ ]:
# SVD：A = U Σ Vᵀ —— PCA / 推荐系统 / 图像压缩 都用它
# SVD: foundation of PCA, recommenders, image compression
A = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 9.],
              [10., 11., 12.]])
U, s, Vt = np.linalg.svd(A, full_matrices=False)
print("U shape :", U.shape)         # (4, 3)
print("s shape :", s.shape)         # (3,) 奇异值（向量）/ singular values
print("Vt shape:", Vt.shape)        # (3, 3)

# 重构 / Reconstruct
A_recon = U @ np.diag(s) @ Vt
print("\nrecon ≈ A ?", np.allclose(A_recon, A))


<a id="12"></a>
## 12. 随机数生成 / Random Number Generation

DS 里几乎所有"模拟"和"洗牌"动作都靠这个。
Everything stochastic in DS uses this.

**只用 `np.random.default_rng()`**，老 API 不要用（全局状态、不可复现）。
**Use only `np.random.default_rng()`** — the legacy API has global state and reproducibility issues.


In [ ]:
rng = np.random.default_rng(seed=2026)

# 各种分布 / Distributions
print("uniform [0,1)     :", rng.random(3))
print("uniform [-1, 1)   :", rng.uniform(-1, 1, size=3))
print("normal N(μ=0,σ=1) :", rng.standard_normal(3))
print("normal N(μ=10,σ=2):", rng.normal(loc=10, scale=2, size=3))
print("Bernoulli p=0.3   :", rng.binomial(n=1, p=0.3, size=10))
print("Poisson λ=3       :", rng.poisson(lam=3, size=10))
print("multinomial       :", rng.multinomial(n=10, pvals=[0.2, 0.3, 0.5]))


In [ ]:
# 常见动作：从数组里随机抽样 / Sample without replacement
data = np.arange(100)
sampled = rng.choice(data, size=10, replace=False)
print("sampled (no replace):", sampled)

# 洗牌 / Shuffle
arr = np.arange(10)
rng.shuffle(arr)        # 原地 / in-place
print("shuffled :", arr)

# 划分 train/test ——一行 / Train-test split in one line
X = np.arange(20).reshape(10, 2)
y = np.arange(10)
perm = rng.permutation(len(X))
n_train = int(0.7 * len(X))
X_tr, X_te = X[perm[:n_train]], X[perm[n_train:]]
y_tr, y_te = y[perm[:n_train]], y[perm[n_train:]]
print(f"\ntrain size: {len(X_tr)}, test size: {len(X_te)}")


<a id="13"></a>
## 13. `einsum` —— 张量运算瑞士军刀 / The Tensor Swiss Army Knife

`einsum` = Einstein summation。一个字符串描述任意张量运算。**深度学习论文里到处都是**（attention、convolution 都可以写成 einsum）。
`einsum` describes any tensor op with a string. **Ubiquitous in DL papers** — attention, conv all expressible as einsum.

规则：
- 字符串左侧是输入下标，右侧是输出下标
- **重复出现且不在输出中的下标 → 求和**
- 出现且**在输出中 → 保留**

例：矩阵乘 $C_{ik} = \sum_j A_{ij} B_{jk}$ → `"ij,jk->ik"`


In [ ]:
A = np.arange(6).reshape(2, 3)
B = np.arange(12).reshape(3, 4)

# 矩阵乘 / Matmul
C1 = A @ B
C2 = np.einsum("ij,jk->ik", A, B)
print("matmul same?", np.allclose(C1, C2))


In [ ]:
# 几个常见 einsum 模式 / Common einsum patterns
x = np.arange(5)
y = np.arange(5)

# 内积 / inner product: x·y = Σ xᵢ yᵢ
print("inner    :", np.einsum("i,i->", x, y))    # 标量

# 外积 / outer product: M_{ij} = x_i y_j
print("outer:\n", np.einsum("i,j->ij", x, y))

# 转置 / transpose
A = np.arange(6).reshape(2, 3)
print("\ntransposed:\n", np.einsum("ij->ji", A))

# 沿某轴求和 / sum along axis
print("\nrow sums :", np.einsum("ij->i", A))
print("col sums :", np.einsum("ij->j", A))
print("total    :", np.einsum("ij->", A))

# 元素积后求和（等价 (A*B).sum）/ Hadamard then sum
B = np.arange(6).reshape(2, 3)
print("\nelementwise sum:", np.einsum("ij,ij->", A, B))


更复杂的 / Heavier example: **batch matrix multiply**

数学：对一批 $B$ 个矩阵对做矩阵乘，$\mathbf{C}^{(k)} = \mathbf{A}^{(k)} \mathbf{B}^{(k)}$
Math: for a batch of $B$ matrix pairs, multiply each pair.


In [ ]:
# A: (B, m, n), B: (B, n, p) -> C: (B, m, p)
rng = np.random.default_rng(0)
A = rng.normal(size=(4, 2, 3))
B = rng.normal(size=(4, 3, 5))

C = np.einsum("bij,bjk->bik", A, B)
print("batched matmul shape:", C.shape)         # (4, 2, 5)

# 等价于 / Equivalent to:
print("equals A @ B ?", np.allclose(C, A @ B))   # @ 也支持 batch / @ supports batch too


<a id="14"></a>
## 14. 性能小贴士 / Performance Tips

1. **能向量化就向量化，绝不写 Python for 循环**
   Vectorize everything; never write Python loops on arrays.
2. **避免不必要的拷贝**：`view` 比 `copy` 快很多
   Avoid copies; views are far cheaper.
3. **`np.ascontiguousarray`**：保证 C 连续 → BLAS 才能跑满速度
   Ensures C-contiguous layout so BLAS can vectorize.
4. **预分配**：循环里要追加结果时，**先开好数组**，不要 `np.append` 在循环里调（每次都拷贝）
   Pre-allocate; never `np.append` in a loop — it copies every iteration.
5. **`float32` 比 `float64` 快**（内存带宽减半，SIMD 多 2 倍并行）；可接受精度损失时优先用
   Prefer `float32` when precision allows — half the memory bandwidth, 2× SIMD lanes.


In [ ]:
import time

# 反例：np.append 在循环里 / Anti-pattern: np.append in loop
n = 100_000

t0 = time.perf_counter()
out_bad = np.array([], dtype=np.float64)
for i in range(n):
    out_bad = np.append(out_bad, i * 0.5)
t_bad = time.perf_counter() - t0

# 正例：预分配 / Correct: pre-allocate
t0 = time.perf_counter()
out_good = np.empty(n)
for i in range(n):
    out_good[i] = i * 0.5
t_good = time.perf_counter() - t0

# 最佳：纯向量化 / Best: vectorize
t0 = time.perf_counter()
out_best = np.arange(n) * 0.5
t_best = time.perf_counter() - t0

print(f"np.append in loop : {t_bad*1000:7.1f} ms")
print(f"pre-allocate      : {t_good*1000:7.1f} ms")
print(f"vectorized        : {t_best*1000:7.1f} ms")


<a id="15"></a>
## 15. 实战：用 NumPy 在 California Housing 上做线性回归 / Hands-on

> **🏠 数据集介绍 / Dataset Intro: California Housing**
>
> **来源 / Source**: Pace, R. Kelley and Ronald Barry (1997). *Sparse spatial autoregressions*. 现内置于 `sklearn.datasets.fetch_california_housing()`。
> Built into `sklearn.datasets.fetch_california_housing()`.
>
> **背景 / Background**: 加州 1990 年人口普查数据。**Boston Housing 数据集因伦理问题被 sklearn 弃用后**，这个数据集成为线性回归的标准教学集。
> California 1990 census data. **Replaced Boston Housing in sklearn** (which was deprecated for ethical reasons) as the standard regression teaching set.
>
> | 字段 / Field | 含义 / Meaning |
> |---|---|
> | `MedInc`     | 街区中位收入（万美元）/ median income ($10k) |
> | `HouseAge`   | 房屋中位年龄 / median house age |
> | `AveRooms`   | 平均房间数 / average rooms per dwelling |
> | `AveBedrms`  | 平均卧室数 / average bedrooms per dwelling |
> | `Population` | 街区人口 / block population |
> | `AveOccup`   | 平均入住人数 / average household occupancy |
> | `Latitude`   | 纬度 / latitude |
> | `Longitude`  | 经度 / longitude |
> | **target**: `MedHouseVal` | 街区房屋中位价（10 万美元）/ median house value ($100k) |
>
> **规模 / Size**: 20,640 行，8 特征。
> 20,640 rows, 8 features.
>
> **任务 / Task**: 回归 / regression — predict `MedHouseVal`.


In [ ]:
# 1) 加载数据 / Load
from sklearn.datasets import fetch_california_housing

dataset = fetch_california_housing()
X = dataset.data            # (20640, 8)  numpy array
y = dataset.target          # (20640,)
feature_names = dataset.feature_names

print(f"X shape       : {X.shape}")
print(f"y shape       : {y.shape}")
print(f"features      : {feature_names}")
print(f"X dtype       : {X.dtype}")


In [ ]:
# 2) 数据集"巡视"——用纯 NumPy 做 EDA
# Quick EDA — pure NumPy
print(f"#samples (n)  : {X.shape[0]:,}")
print(f"#features (d) : {X.shape[1]}")

# 一张表：每个特征的 min / mean / std / max
# Per-feature stats
print(f"\n{'feature':<12} {'min':>10} {'mean':>10} {'std':>10} {'max':>10}")
print("-" * 56)
for j, name in enumerate(feature_names):
    col = X[:, j]
    print(f"{name:<12} {col.min():>10.3f} {col.mean():>10.3f} "
          f"{col.std():>10.3f} {col.max():>10.3f}")

print(f"\ntarget y      : min={y.min():.3f}  mean={y.mean():.3f}  "
      f"std={y.std():.3f}  max={y.max():.3f}")


**观察 / Observations**：
- 特征**量纲差异巨大**（`Population` 上千，`AveBedrms` 个位数）→ 必须**标准化**才能让线性回归正常收敛。
  Features are on **wildly different scales** → must standardize for stable regression.
- `y` 的最大值是 5.0 ——其实数据里被**截顶**了（原始数据里 > 5 的都被压成 5），这是个**经典数据陷阱**。
  `y` is capped at 5.0 — a classic data trap.


In [ ]:
# 3) 标准化：每列减均值除标准差 / Z-score normalize each column
# Math:
#   X'_{ij} = (X_{ij} - μ_j) / σ_j
mu = X.mean(axis=0, keepdims=True)     # (1, 8)
sigma = X.std(axis=0, keepdims=True)   # (1, 8)
X_std = (X - mu) / sigma                # 广播 / broadcasting

# 加一列 1 作为 bias 项 / append a column of 1s for the intercept
n = X_std.shape[0]
X_aug = np.hstack([np.ones((n, 1)), X_std])    # (n, 9)
print(f"X_aug shape: {X_aug.shape}")


### 线性回归数学 / The Math of Linear Regression

模型 / Model:
$$\hat{y}_i = w_0 + \sum_{j=1}^{d} w_j\, x_{ij} = \mathbf{w}^\top \tilde{\mathbf{x}}_i$$

其中 $\tilde{\mathbf{x}}_i = (1, x_{i1}, \dots, x_{id})^\top$（前面加了 1 用于 bias）。
Where $\tilde{\mathbf{x}}_i$ has a 1 prepended for the bias.

经验风险（MSE）/ Empirical risk (MSE):
$$J(\mathbf{w}) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \mathbf{w}^\top \tilde{\mathbf{x}}_i)^2 = \frac{1}{n} \|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2$$

令梯度为 0：
$$\nabla_{\mathbf{w}} J = -\frac{2}{n} \mathbf{X}^\top (\mathbf{y} - \mathbf{X}\mathbf{w}) = \mathbf{0}$$

得**正规方程 / Normal Equation**:
$$\boxed{\; \hat{\mathbf{w}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y} \;}$$

但**别真用 `inv`**！用 `np.linalg.solve(X.T @ X, X.T @ y)`，数值更稳。
**Don't use `inv` literally** — use `solve(X.T @ X, X.T @ y)` for numerical stability.


In [ ]:
# 4) 用正规方程解 / Solve the normal equation
# w_hat = (XᵀX)^{-1} Xᵀy  -- 但用 solve 而不是 inv
XtX = X_aug.T @ X_aug              # (9, 9)
Xty = X_aug.T @ y                  # (9,)
w_hat = np.linalg.solve(XtX, Xty)  # (9,)

print(f"intercept (w_0): {w_hat[0]:.4f}")
print(f"\n{'feature':<12} {'coefficient':>14}")
print("-" * 28)
for name, w in zip(feature_names, w_hat[1:]):
    print(f"{name:<12} {w:>14.4f}")


In [ ]:
# 5) 在训练集上预测并算 R²
# Predict and compute R²
y_pred = X_aug @ w_hat
residuals = y - y_pred

# Math:
#   R² = 1 - SS_res / SS_tot
#   SS_res = Σ (yᵢ - ŷᵢ)²
#   SS_tot = Σ (yᵢ - ȳ)²
ss_res = (residuals ** 2).sum()
ss_tot = ((y - y.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot
rmse = np.sqrt((residuals ** 2).mean())

print(f"RMSE  : {rmse:.4f} ($100k units)")
print(f"R²    : {r2:.4f}")


In [ ]:
# 6) 和 sklearn 的实现对照 / Cross-check against sklearn
# 只是为了验证我们手写的实现正确——不是"作弊"
from sklearn.linear_model import LinearRegression
ref = LinearRegression().fit(X_std, y)
print("sklearn intercept :", ref.intercept_.round(4))
print("our intercept     :", w_hat[0].round(4))

print("\nmax |coef difference|:",
      np.abs(ref.coef_ - w_hat[1:]).max())


**两组系数完全一致**（差异在 1e-12 量级，是浮点误差）——我们用 NumPy 复现了 sklearn 的 `LinearRegression`。
**Coefficients match** to floating-point precision — we replicated `LinearRegression` with bare NumPy.

到这里你应该明白：**`LinearRegression` 的本质就是几行 NumPy + 一道线性代数题**。
The takeaway: **`LinearRegression` is just a few lines of NumPy + one linear-algebra problem**.

注：后面 Part 4.1 我们会**正式**做线性回归（含残差诊断、显著性、置信区间）。本节只是 NumPy 综合演练。
We'll cover linear regression **formally** in Part 4.1 (residuals, significance, CIs). This was a NumPy showcase.


<a id="16"></a>
## 16. 小结 / Summary

| 主题 / Topic | 关键 / Key takeaway |
|---|---|
| `ndarray` 解剖 | shape / dtype / strides 决定一切 |
| 创建数组 | `np.array`, `arange`, `linspace`, `default_rng()` |
| 索引切片 | basic = 视图；fancy / boolean = 拷贝 |
| 广播 | 右对齐，1 维自动扩展；最常踩 `(n,d)+(n,)` 的坑 |
| ufunc | 逐元素 + 广播 + C 实现 |
| 聚合 + axis | `axis=k` ⇒ 压扁第 k 维；`keepdims=True` 救命 |
| 形状操作 | `reshape(-1, …)`, `vstack/hstack`, `T`, `swapaxes` |
| 线性代数 | 永远 `solve` 不 `inv`；`@` 优先；`svd/eigh` 是 PCA 地基 |
| 随机数 | 只用 `default_rng(seed)` |
| `einsum` | 攻击任何张量运算的终极武器 |
| 性能 | 向量化、预分配、float32、连续内存 |
| 实战 | 正规方程 = 一行 `solve(XᵀX, Xᵀy)` |

### 工业场景速查 / Cheat sheet

- **标准化**：`(X - X.mean(0)) / X.std(0)` — 一行
- **加 bias 列**：`np.hstack([np.ones((n, 1)), X])`
- **训练/验证划分**：`rng.permutation(n)` + 切片
- **解线性方程**：`np.linalg.solve(A, b)`
- **求特征值**（对称矩阵，如协方差）：`np.linalg.eigh(C)`
- **SVD（PCA、推荐）**：`np.linalg.svd(X, full_matrices=False)`
- **找极值的位置**：`argmax / argmin`，配 `axis=`
- **批量矩阵乘**：`A @ B`（自动 batch）或 `einsum("bij,bjk->bik", A, B)`

### 下一节预告 / Next up

**Part 0.3 · Pandas 全面解析** —— 把 NumPy 包装成"自带列名"的数据表。我们会用 Titanic 数据集系统地走一遍 `Series` / `DataFrame` / `groupby` / `merge` / `pivot`。
**Part 0.3 · Pandas Deep Dive** — NumPy with column names. We'll walk through `Series` / `DataFrame` / `groupby` / `merge` / `pivot` on the Titanic dataset.
